# Homework 4. DML, DDL and Advanced SQL
## Task 1. Setup and Olist dataset

In [1]:
%pip install -q pgserver psycopg2-binary sqlalchemy pandas kagglehub

import os
import pandas as pd
import pgserver
from sqlalchemy import create_engine, text

# Start PostgreSQL 16
pg = pgserver.get_server('/tmp/hw4_pg', cleanup_mode='stop')
engine = create_engine(pg.get_uri(), future=True)

# Check PostgreSQL version
with engine.connect() as conn:
    version = conn.execute(text("SELECT version()")).scalar_one()
    print(version)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 28.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 28.9 MB/s eta 0:00:00
PostgreSQL 16.2 on x86_64-pc-linux-gnu, compiled by gcc (GCC) 10.2.1 20210130 (Red Hat 10.2.1-11), 64-bit


In [2]:
import kagglehub

# Download the Olist dataset from Kaggle
dataset_dir = kagglehub.dataset_download(
    "olistbr/brazilian-ecommerce"
)

print("Dataset directory:", dataset_dir)

# Check downloaded files
print("\nAvailable files:")
for filename in sorted(os.listdir(dataset_dir)):
    print(filename)

Using Colab cache for faster access to the 'brazilian-ecommerce' dataset.
Dataset directory: /kaggle/input/brazilian-ecommerce

Available files:
olist_customers_dataset.csv
olist_geolocation_dataset.csv
olist_order_items_dataset.csv
olist_order_payments_dataset.csv
olist_order_reviews_dataset.csv
olist_orders_dataset.csv
olist_products_dataset.csv
olist_sellers_dataset.csv
product_category_name_translation.csv


In [3]:
# Map PostgreSQL raw tables to CSV files
files = {
    "olist_customers_raw": "olist_customers_dataset.csv",
    "olist_orders_raw": "olist_orders_dataset.csv",
    "olist_order_items_raw": "olist_order_items_dataset.csv",
    "olist_products_raw": "olist_products_dataset.csv",
    "olist_sellers_raw": "olist_sellers_dataset.csv",
    "olist_order_reviews_raw": "olist_order_reviews_dataset.csv",
}

# Load CSV files into raw PostgreSQL tables
for table_name, file_name in files.items():
    path = os.path.join(dataset_dir, file_name)

    df = pd.read_csv(path)

    df.to_sql(
        table_name,
        engine,
        if_exists="replace",
        index=False,
        chunksize=10_000
    )

    print(f"{table_name}: {df.shape[0]} rows, {df.shape[1]} columns")

olist_customers_raw: 99441 rows, 5 columns
olist_orders_raw: 99441 rows, 8 columns
olist_order_items_raw: 112650 rows, 7 columns
olist_products_raw: 32951 rows, 9 columns
olist_sellers_raw: 3095 rows, 4 columns
olist_order_reviews_raw: 99224 rows, 7 columns


In [4]:
# Verify row counts in all raw tables
df = pd.read_sql(text("""
    SELECT 'customers' AS table_name, COUNT(*) AS n_rows
    FROM olist_customers_raw

    UNION ALL
    SELECT 'orders', COUNT(*) FROM olist_orders_raw

    UNION ALL
    SELECT 'order_items', COUNT(*) FROM olist_order_items_raw

    UNION ALL
    SELECT 'products', COUNT(*) FROM olist_products_raw

    UNION ALL
    SELECT 'sellers', COUNT(*) FROM olist_sellers_raw

    UNION ALL
    SELECT 'reviews', COUNT(*) FROM olist_order_reviews_raw

    ORDER BY table_name
"""), engine)

display(df)

,table_name,n_rows
0,customers,99441
1,order_items,112650
2,orders,99441
3,products,32951
4,reviews,99224
5,sellers,3095


In [5]:
# Create typed tables with keys and constraints
typed_ddl = """
DROP TABLE IF EXISTS
    olist_order_reviews_typed,
    olist_order_items_typed,
    olist_orders_typed,
    olist_products_typed,
    olist_sellers_typed,
    olist_customers_typed
CASCADE;

CREATE TABLE olist_customers_typed (
    customer_id TEXT PRIMARY KEY,
    customer_unique_id TEXT NOT NULL,
    customer_zip_code_prefix INTEGER,
    customer_city TEXT,
    customer_state TEXT
);

CREATE TABLE olist_products_typed (
    product_id TEXT PRIMARY KEY,
    product_category_name TEXT,
    product_name_length INTEGER,
    product_description_length INTEGER,
    product_photos_qty INTEGER,
    product_weight_g INTEGER,
    product_length_cm INTEGER,
    product_height_cm INTEGER,
    product_width_cm INTEGER,

    CHECK (product_weight_g IS NULL OR product_weight_g >= 0),
    CHECK (product_photos_qty IS NULL OR product_photos_qty >= 0)
);

CREATE TABLE olist_sellers_typed (
    seller_id TEXT PRIMARY KEY,
    seller_zip_code_prefix INTEGER,
    seller_city TEXT,
    seller_state TEXT
);

CREATE TABLE olist_orders_typed (
    order_id TEXT PRIMARY KEY,
    customer_id TEXT NOT NULL
        REFERENCES olist_customers_typed(customer_id),
    order_status TEXT NOT NULL,
    order_purchase_timestamp TIMESTAMP,
    order_approved_at TIMESTAMP,
    order_delivered_carrier_date TIMESTAMP,
    order_delivered_customer_date TIMESTAMP,
    order_estimated_delivery_date TIMESTAMP
);

CREATE TABLE olist_order_items_typed (
    order_id TEXT NOT NULL
        REFERENCES olist_orders_typed(order_id),
    order_item_id INTEGER NOT NULL,
    product_id TEXT NOT NULL
        REFERENCES olist_products_typed(product_id),
    seller_id TEXT NOT NULL
        REFERENCES olist_sellers_typed(seller_id),
    shipping_limit_date TIMESTAMP,
    price NUMERIC(12, 2) NOT NULL,
    freight_value NUMERIC(12, 2) NOT NULL,

    PRIMARY KEY (order_id, order_item_id),

    CHECK (order_item_id > 0),
    CHECK (price >= 0),
    CHECK (freight_value >= 0)
);

CREATE TABLE olist_order_reviews_typed (
    review_row_id BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    review_id TEXT NOT NULL,
    order_id TEXT NOT NULL
        REFERENCES olist_orders_typed(order_id),
    review_score INTEGER NOT NULL
        CHECK (review_score BETWEEN 1 AND 5),
    review_comment_title TEXT,
    review_comment_message TEXT,
    review_creation_date TIMESTAMP,
    review_answer_timestamp TIMESTAMP
);

CREATE INDEX idx_olist_customer_unique
    ON olist_customers_typed(customer_unique_id);

CREATE INDEX idx_olist_items_seller
    ON olist_order_items_typed(seller_id);

CREATE INDEX idx_olist_reviews_order
    ON olist_order_reviews_typed(order_id);
"""

with engine.begin() as conn:
    conn.execute(text(typed_ddl))

print("Typed tables created successfully.")

Typed tables created successfully.


In [6]:
# Transfer raw data to typed tables
insert_typed_sql = """
INSERT INTO olist_customers_typed
SELECT
    customer_id,
    customer_unique_id,
    customer_zip_code_prefix::INTEGER,
    customer_city,
    customer_state
FROM olist_customers_raw;

INSERT INTO olist_products_typed
SELECT
    product_id,
    product_category_name,
    product_name_lenght::INTEGER,
    product_description_lenght::INTEGER,
    product_photos_qty::INTEGER,
    product_weight_g::INTEGER,
    product_length_cm::INTEGER,
    product_height_cm::INTEGER,
    product_width_cm::INTEGER
FROM olist_products_raw;

INSERT INTO olist_sellers_typed
SELECT
    seller_id,
    seller_zip_code_prefix::INTEGER,
    seller_city,
    seller_state
FROM olist_sellers_raw;

INSERT INTO olist_orders_typed
SELECT
    order_id,
    customer_id,
    order_status,
    order_purchase_timestamp::TIMESTAMP,
    order_approved_at::TIMESTAMP,
    order_delivered_carrier_date::TIMESTAMP,
    order_delivered_customer_date::TIMESTAMP,
    order_estimated_delivery_date::TIMESTAMP
FROM olist_orders_raw;

INSERT INTO olist_order_items_typed
SELECT
    order_id,
    order_item_id::INTEGER,
    product_id,
    seller_id,
    shipping_limit_date::TIMESTAMP,
    price::NUMERIC(12, 2),
    freight_value::NUMERIC(12, 2)
FROM olist_order_items_raw;

INSERT INTO olist_order_reviews_typed (
    review_id,
    order_id,
    review_score,
    review_comment_title,
    review_comment_message,
    review_creation_date,
    review_answer_timestamp
)
SELECT
    review_id,
    order_id,
    review_score::INTEGER,
    review_comment_title,
    review_comment_message,
    review_creation_date::TIMESTAMP,
    review_answer_timestamp::TIMESTAMP
FROM olist_order_reviews_raw;
"""

with engine.begin() as conn:
    conn.execute(text(insert_typed_sql))

print("Typed data inserted successfully.")

Typed data inserted successfully.


In [7]:
# Compare row counts in raw and typed tables
df = pd.read_sql(text("""
    SELECT 'customers' AS table_name,
           (SELECT COUNT(*) FROM olist_customers_raw) AS raw_rows,
           (SELECT COUNT(*) FROM olist_customers_typed) AS typed_rows

    UNION ALL
    SELECT 'orders',
           (SELECT COUNT(*) FROM olist_orders_raw),
           (SELECT COUNT(*) FROM olist_orders_typed)

    UNION ALL
    SELECT 'order_items',
           (SELECT COUNT(*) FROM olist_order_items_raw),
           (SELECT COUNT(*) FROM olist_order_items_typed)

    UNION ALL
    SELECT 'products',
           (SELECT COUNT(*) FROM olist_products_raw),
           (SELECT COUNT(*) FROM olist_products_typed)

    UNION ALL
    SELECT 'sellers',
           (SELECT COUNT(*) FROM olist_sellers_raw),
           (SELECT COUNT(*) FROM olist_sellers_typed)

    UNION ALL
    SELECT 'reviews',
           (SELECT COUNT(*) FROM olist_order_reviews_raw),
           (SELECT COUNT(*) FROM olist_order_reviews_typed)

    ORDER BY table_name
"""), engine)

display(df)

,table_name,raw_rows,typed_rows
0,customers,99441,99441
1,order_items,112650,112650
2,orders,99441,99441
3,products,32951,32951
4,reviews,99224,99224
5,sellers,3095,3095


In [8]:
# Check foreign key relationships in order items
df = pd.read_sql(text("""
    SELECT
        COUNT(*) AS total_items,
        COUNT(*) FILTER (
            WHERE o.order_id IS NULL
        ) AS missing_orders,
        COUNT(*) FILTER (
            WHERE p.product_id IS NULL
        ) AS missing_products,
        COUNT(*) FILTER (
            WHERE s.seller_id IS NULL
        ) AS missing_sellers
    FROM olist_order_items_typed AS i
    LEFT JOIN olist_orders_typed AS o
        ON i.order_id = o.order_id
    LEFT JOIN olist_products_typed AS p
        ON i.product_id = p.product_id
    LEFT JOIN olist_sellers_typed AS s
        ON i.seller_id = s.seller_id
"""), engine)

display(df)

,total_items,missing_orders,missing_products,missing_sellers
0,112650,0,0,0


## Task 2. DML Operations in PostgreSQL

In [9]:
# Create a separate table for testing DML operations
dml_ddl = """
DROP TABLE IF EXISTS hw4_seller_notes CASCADE;

CREATE TABLE hw4_seller_notes (
    note_id BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    seller_id TEXT NOT NULL UNIQUE
        REFERENCES olist_sellers_typed(seller_id),
    note TEXT NOT NULL,
    seller_city TEXT,
    status TEXT NOT NULL DEFAULT 'New'
        CHECK (status IN ('New', 'Reviewed', 'Updated')),
    updated_at TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP
);
"""

with engine.begin() as conn:
    conn.execute(text(dml_ddl))

print("DML test table created successfully.")

DML test table created successfully.


In [10]:
# Insert notes for two existing sellers and return inserted rows
insert_sql = """
INSERT INTO hw4_seller_notes (seller_id, note)
SELECT
    seller_id,
    'Initial seller review'
FROM olist_sellers_typed
ORDER BY seller_id
LIMIT 2
RETURNING note_id, seller_id, note, status;
"""

with engine.begin() as conn:
    result = conn.execute(text(insert_sql))
    inserted_df = pd.DataFrame(result.mappings().all())

display(inserted_df)

,note,note_id,seller_id,status
0,Initial seller review,1,0015a82c2db000af6aaaf3ae2ecb0532,New
1,Initial seller review,2,001cca7ae9ae17fb1caed9dfb1094831,New


In [11]:
# Update an existing seller note using UPSERT
upsert_sql = """
INSERT INTO hw4_seller_notes (
    seller_id,
    note,
    status
)
SELECT
    seller_id,
    'Seller checked during ETL',
    'Reviewed'
FROM olist_sellers_typed
ORDER BY seller_id
LIMIT 1

ON CONFLICT (seller_id)
DO UPDATE SET
    note = EXCLUDED.note,
    status = EXCLUDED.status,
    updated_at = CURRENT_TIMESTAMP

RETURNING note_id, seller_id, note, status;
"""

with engine.begin() as conn:
    result = conn.execute(text(upsert_sql))
    upsert_df = pd.DataFrame(result.mappings().all())

display(upsert_df)

,note,note_id,seller_id,status
0,Seller checked during ETL,1,0015a82c2db000af6aaaf3ae2ecb0532,Reviewed


In [12]:
# Update seller notes using data from the sellers table
update_sql = """
UPDATE hw4_seller_notes AS n
SET
    seller_city = s.seller_city,
    status = 'Updated',
    updated_at = CURRENT_TIMESTAMP
FROM olist_sellers_typed AS s
WHERE n.seller_id = s.seller_id

RETURNING
    n.note_id,
    n.seller_id,
    n.seller_city,
    n.status;
"""

with engine.begin() as conn:
    result = conn.execute(text(update_sql))
    updated_df = pd.DataFrame(result.mappings().all())

display(updated_df)

,note_id,seller_city,seller_id,status
0,2,cariacica,001cca7ae9ae17fb1caed9dfb1094831,Updated
1,1,santo andre,0015a82c2db000af6aaaf3ae2ecb0532,Updated


In [13]:
# Delete one test record and return the deleted row
delete_sql = """
DELETE FROM hw4_seller_notes
WHERE note_id = (
    SELECT MAX(note_id)
    FROM hw4_seller_notes
)
RETURNING
    note_id,
    seller_id,
    note,
    status;
"""

with engine.begin() as conn:
    result = conn.execute(text(delete_sql))
    deleted_df = pd.DataFrame(result.mappings().all())

display(deleted_df)

,note,note_id,seller_id,status
0,Initial seller review,2,001cca7ae9ae17fb1caed9dfb1094831,Updated


In [14]:
# Verify the final state of the DML test table
df = pd.read_sql(text("""
    SELECT
        note_id,
        seller_id,
        note,
        seller_city,
        status
    FROM hw4_seller_notes
    ORDER BY note_id
"""), engine)

display(df)

assert len(df) == 1, "Expected exactly one remaining record"
assert df.iloc[0]["status"] == "Updated"
assert df.iloc[0]["note"] == "Seller checked during ETL"

print("DML verification passed.")

,note_id,seller_id,note,seller_city,status
0,1,0015a82c2db000af6aaaf3ae2ecb0532,Seller checked during ETL,santo andre,Updated


DML verification passed.


### Висновок

У цій частині було виконано чотири DML-операції на окремій тестовій таблиці.

`INSERT ... RETURNING` дозволив додати записи та одразу переглянути результат. `ON CONFLICT` було використано для оновлення наявного запису без створення дубліката.

За допомогою `UPDATE ... FROM` було додано інформацію з іншої таблиці, а `DELETE ... RETURNING` дозволив видалити тестовий запис і побачити, які саме дані були видалені.

Усі операції виконувалися в окремих транзакціях. Початкові таблиці Olist не змінювалися.

Такий підхід дозволяє безпечніше працювати з даними та контролювати результати змін.

## Task 3. Customer Segmentation

In [15]:
# Create a dimension table with one row per unique customer
customer_dim_sql = """
DROP TABLE IF EXISTS customer_segments CASCADE;
DROP TABLE IF EXISTS olist_customer_dim CASCADE;

CREATE TABLE olist_customer_dim (
    customer_unique_id TEXT PRIMARY KEY,
    latest_customer_id TEXT
        REFERENCES olist_customers_typed(customer_id),
    customer_state CHAR(2),
    customer_city TEXT
);

INSERT INTO olist_customer_dim (
    customer_unique_id,
    latest_customer_id,
    customer_state,
    customer_city
)
SELECT DISTINCT ON (c.customer_unique_id)
    c.customer_unique_id,
    c.customer_id,
    c.customer_state,
    c.customer_city
FROM olist_customers_typed AS c
LEFT JOIN olist_orders_typed AS o
    ON o.customer_id = c.customer_id
ORDER BY
    c.customer_unique_id,
    o.order_purchase_timestamp DESC NULLS LAST,
    c.customer_id;
"""

with engine.begin() as conn:
    conn.execute(text(customer_dim_sql))

print("Customer dimension created successfully.")

Customer dimension created successfully.


In [16]:
# Compare unique customers with customer dimension rows
df = pd.read_sql(text("""
    SELECT
        (SELECT COUNT(DISTINCT customer_unique_id)
         FROM olist_customers_typed) AS unique_customers,

        (SELECT COUNT(*)
         FROM olist_customer_dim) AS dimension_rows
"""), engine)

display(df)

,unique_customers,dimension_rows
0,96096,96096


In [17]:
# Create customer segmentation table with constraints
segments_ddl = """
CREATE TABLE customer_segments (
    segment_id BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,

    customer_unique_id TEXT NOT NULL,

    segment_name TEXT NOT NULL
        CHECK (segment_name IN ('VIP', 'regular', 'new', 'inactive')),

    rfm_score SMALLINT NOT NULL
        CHECK (rfm_score BETWEEN 1 AND 5),

    recency_days INTEGER
        CHECK (recency_days IS NULL OR recency_days >= 0),

    monetary_value NUMERIC(12, 2) NOT NULL DEFAULT 0
        CHECK (monetary_value >= 0),

    n_orders INTEGER NOT NULL DEFAULT 0
        CHECK (n_orders >= 0),

    assigned_at TIMESTAMPTZ NOT NULL DEFAULT NOW(),

    UNIQUE (customer_unique_id)
);

ALTER TABLE customer_segments
    ADD CONSTRAINT fk_customer_segments_customer_unique
    FOREIGN KEY (customer_unique_id)
    REFERENCES olist_customer_dim(customer_unique_id)
    ON DELETE CASCADE;
"""

with engine.begin() as conn:
    conn.execute(text(segments_ddl))

print("Customer segments table created successfully.")

Customer segments table created successfully.


In [18]:
# Calculate customer segments using historical order data
segmentation_sql = """
WITH reference_date AS (
    SELECT
        (MAX(order_purchase_timestamp)::DATE + 1) AS as_of_date
    FROM olist_orders_typed
),

order_totals AS (
    SELECT
        o.order_id,
        o.customer_id,
        o.order_purchase_timestamp::DATE AS order_date,
        COALESCE(
            SUM(oi.price + oi.freight_value), 0
        )::NUMERIC(12, 2) AS order_total
    FROM olist_orders_typed AS o
    LEFT JOIN olist_order_items_typed AS oi
        ON oi.order_id = o.order_id
    GROUP BY
        o.order_id,
        o.customer_id,
        o.order_purchase_timestamp
),

customer_stats AS (
    SELECT
        c.customer_unique_id,
        COUNT(DISTINCT ot.order_id)::INTEGER AS n_orders,
        COALESCE(
            SUM(ot.order_total), 0
        )::NUMERIC(12, 2) AS monetary_value,
        MAX(ot.order_date) AS last_order_date
    FROM olist_customer_dim AS cd
    JOIN olist_customers_typed AS c
        ON c.customer_unique_id = cd.customer_unique_id
    LEFT JOIN order_totals AS ot
        ON ot.customer_id = c.customer_id
    GROUP BY c.customer_unique_id
),

segmented AS (
    SELECT
        cs.customer_unique_id,
        cs.n_orders,
        cs.monetary_value,

        CASE
            WHEN cs.last_order_date IS NULL THEN NULL
            ELSE (rd.as_of_date - cs.last_order_date)::INTEGER
        END AS recency_days,

        CASE
            WHEN cs.monetary_value >= 1000 THEN 'VIP'

            WHEN cs.last_order_date IS NOT NULL
                 AND (rd.as_of_date - cs.last_order_date)::INTEGER <= 90
                 AND cs.n_orders = 1
                THEN 'new'

            WHEN cs.n_orders >= 2
                 OR cs.monetary_value >= 100
                THEN 'regular'

            ELSE 'inactive'
        END AS segment_name,

        CASE
            WHEN cs.monetary_value >= 1000 THEN 5
            WHEN cs.monetary_value >= 500 THEN 4
            WHEN cs.n_orders >= 2 THEN 3
            WHEN cs.n_orders = 1 THEN 2
            ELSE 1
        END::SMALLINT AS rfm_score

    FROM customer_stats AS cs
    CROSS JOIN reference_date AS rd
)

INSERT INTO customer_segments (
    customer_unique_id,
    segment_name,
    rfm_score,
    recency_days,
    monetary_value,
    n_orders
)
SELECT
    customer_unique_id,
    segment_name,
    rfm_score,
    recency_days,
    monetary_value,
    n_orders
FROM segmented

ON CONFLICT (customer_unique_id)
DO UPDATE SET
    segment_name = EXCLUDED.segment_name,
    rfm_score = EXCLUDED.rfm_score,
    recency_days = EXCLUDED.recency_days,
    monetary_value = EXCLUDED.monetary_value,
    n_orders = EXCLUDED.n_orders,
    assigned_at = NOW()

RETURNING
    customer_unique_id,
    segment_name,
    rfm_score,
    recency_days,
    monetary_value,
    n_orders;
"""

# Execute UPSERT without displaying all returned rows
with engine.begin() as conn:
    result = conn.execute(text(segmentation_sql))
    sample = pd.DataFrame(result.fetchmany(5), columns=result.keys())
    result.close()

print("Customer segmentation completed.")
display(sample)

Customer segmentation completed.


,customer_unique_id,segment_name,rfm_score,recency_days,monetary_value,n_orders
0,0000366f3b9a7992bf8c76cfdf3221e2,regular,2,161,141.90,1
1,0000b849f77a49e4a4ce2b2a4ca5be3f,inactive,2,164,27.19,1
2,0000f46a3911fa3c0805444483337064,inactive,2,587,86.22,1
3,0000f6ccb0745a6a4b88665a16c9f078,inactive,2,371,43.62,1
4,0004aac84e0df4da2b147fca70cf8255,regular,2,338,196.89,1


In [19]:
# Analyze customer distribution across segments
df = pd.read_sql(text("""
    SELECT
        segment_name,
        COUNT(*) AS customer_count,
        ROUND(
            COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (),
            2
        ) AS percentage,
        ROUND(AVG(monetary_value), 2) AS avg_spending,
        ROUND(AVG(n_orders), 2) AS avg_orders
    FROM customer_segments
    GROUP BY segment_name
    ORDER BY customer_count DESC
"""), engine)

display(df)

,segment_name,customer_count,percentage,avg_spending,avg_orders
0,regular,45982,47.85,224.57,1.07
1,inactive,40032,41.66,59.28,1.00
2,new,8891,9.25,140.39,1.00
3,VIP,1191,1.24,1591.81,1.11


In [20]:
# Verify that each unique customer has exactly one segment
df = pd.read_sql(text("""
    SELECT
        COUNT(*) AS total_segments,
        COUNT(DISTINCT customer_unique_id) AS unique_customers,
        COUNT(*) - COUNT(DISTINCT customer_unique_id) AS duplicates
    FROM customer_segments
"""), engine)

display(df)

,total_segments,unique_customers,duplicates
0,96096,96096,0


In [21]:
# Verify customer_segments table constraints
df = pd.read_sql(text("""
    SELECT
        conname,
        contype,
        pg_get_constraintdef(oid) AS definition
    FROM pg_constraint
    WHERE conrelid = 'customer_segments'::regclass
    ORDER BY conname
"""), engine)

display(df)

,conname,contype,definition
0,customer_segments_customer_unique_id_key,u,UNIQUE (customer_unique_id)
1,customer_segments_monetary_value_check,c,CHECK ((monetary_value >= (0)::numeric))
2,customer_segments_n_orders_check,c,CHECK ((n_orders >= 0))
3,customer_segments_pkey,p,PRIMARY KEY (segment_id)
4,customer_segments_recency_days_check,c,CHECK (((recency_days IS NULL) OR (recency_day...
5,customer_segments_rfm_score_check,c,CHECK (((rfm_score >= 1) AND (rfm_score <= 5)))
6,customer_segments_segment_name_check,c,"CHECK ((segment_name = ANY (ARRAY['VIP'::text,..."
7,fk_customer_segments_customer_unique,f,FOREIGN KEY (customer_unique_id) REFERENCES ol...


### Висновок

У результаті сегментації було розподілено 96 096 унікальних покупців на чотири групи.

Найбільша група - `regular`, до якої потрапило 45 982 покупці (47.85%). Ще 41.66% належать до групи `inactive`, а 9.25% - до `new`.

Найменша група - `VIP`, лише 1 191 покупець (1.24%). При цьому середні витрати VIP-покупців становлять 1591.81, що значно більше, ніж в інших групах.

Результати показують, що більшість покупців робили невеликі покупки, а клієнтів із великими витратами відносно мало.

Для сегментації використовувався `customer_unique_id`, тому повторні замовлення одного покупця враховувалися разом.

## Task 4. JOIN Queries

In [22]:
# Query 1: Count orders by customer state using INNER JOIN
q1 = pd.read_sql(text("""
    SELECT
        c.customer_state,
        COUNT(*) AS n_orders
    FROM olist_orders_typed AS o
    INNER JOIN olist_customers_typed AS c
        ON o.customer_id = c.customer_id
    GROUP BY c.customer_state
    ORDER BY n_orders DESC
    LIMIT 10
"""), engine)

display(q1)

top = q1.iloc[0]

print(
    f"Інтерпретація: Найбільше замовлень було зроблено "
    f"зі штату {top['customer_state']} — {top['n_orders']}. "
    "Це показує, що замовлення розподілені між штатами нерівномірно. "
    "Для цього запиту використано INNER JOIN, оскільки потрібні "
    "замовлення, які мають відповідного покупця."
)

,customer_state,n_orders
0,SP,41746
1,RJ,12852
2,MG,11635
3,RS,5466
4,PR,5045
5,SC,3637
6,BA,3380
7,DF,2140
8,ES,2033
9,GO,2020


Інтерпретація: Найбільше замовлень було зроблено зі штату SP — 41746. Це показує, що замовлення розподілені між штатами нерівномірно. Для цього запиту використано INNER JOIN, оскільки потрібні замовлення, які мають відповідного покупця.


In [23]:
# Query 2: Find products with the highest number of sales
q2 = pd.read_sql(text("""
    SELECT
        p.product_id,
        p.product_category_name,
        COUNT(oi.order_id) AS items_sold,
        COALESCE(SUM(oi.price), 0) AS total_sales
    FROM olist_products_typed AS p
    LEFT JOIN olist_order_items_typed AS oi
        ON p.product_id = oi.product_id
    GROUP BY
        p.product_id,
        p.product_category_name
    ORDER BY items_sold DESC, p.product_id
    LIMIT 10
"""), engine)

display(q2)

top = q2.iloc[0]

print(
    f"Інтерпретація: Найбільше проданих позицій має товар "
    f"{top['product_id']} — {top['items_sold']}. "
    f"Сума продажів цього товару становить {top['total_sales']}. "
    "LEFT JOIN також дозволяє враховувати товари без продажів, "
    "хоча вони не потрапляють до першої десятки."
)

,product_id,product_category_name,items_sold,total_sales
0,aca2eb7d00ea1a7b8ebd4e68314663af,moveis_decoracao,527,37608.90
1,99a4788cb24856965c36a24e339b6058,cama_mesa_banho,488,43025.56
2,422879e10f46682990de24d770e7f83d,ferramentas_jardim,484,26577.22
3,389d119b48cf3043d311335e499d9c6b,ferramentas_jardim,392,21440.59
4,368c6c730842d78016ad823897a372db,ferramentas_jardim,388,21056.80
5,53759a2ecddad2bb87a079a1f1519f73,ferramentas_jardim,373,20387.20
6,d1c427060a0f73f6b889a5c7c61f2ac4,informatica_acessorios,343,47214.51
7,53b36df67ebb7c41585e8d54d6772e08,relogios_presentes,323,37683.42
8,154e7e31ebfa092203795c972e5804a6,beleza_saude,281,6325.19
9,3dd2a17168ec895c781a9191c1e95ad7,informatica_acessorios,274,41082.60


Інтерпретація: Найбільше проданих позицій має товар aca2eb7d00ea1a7b8ebd4e68314663af — 527. Сума продажів цього товару становить 37608.9. LEFT JOIN також дозволяє враховувати товари без продажів, хоча вони не потрапляють до першої десятки.


In [24]:
# Query 3: Compare customers and sellers by state
q3 = pd.read_sql(text("""
    WITH customers_by_state AS (
        SELECT
            customer_state AS state,
            COUNT(*) AS n_customers
        FROM olist_customer_dim
        GROUP BY customer_state
    ),
    sellers_by_state AS (
        SELECT
            seller_state AS state,
            COUNT(*) AS n_sellers
        FROM olist_sellers_typed
        GROUP BY seller_state
    )
    SELECT
        COALESCE(c.state, s.state) AS state,
        COALESCE(c.n_customers, 0) AS n_customers,
        COALESCE(s.n_sellers, 0) AS n_sellers
    FROM customers_by_state AS c
    FULL OUTER JOIN sellers_by_state AS s
        ON c.state = s.state
    ORDER BY n_customers DESC, state
    LIMIT 10
"""), engine)

display(q3)

top = q3.iloc[0]

print(
    f"Інтерпретація: Найбільше унікальних покупців "
    f"представлено у штаті {top['state']} — "
    f"{top['n_customers']}. "
    f"У цьому ж штаті зареєстровано {top['n_sellers']} продавців. "
    "FULL OUTER JOIN дозволяє порівняти обидві групи "
    "без втрати штатів, які є лише в одній із таблиць."
)

,state,n_customers,n_sellers
0,SP,40296,1849
1,RJ,12381,171
2,MG,11252,244
3,RS,5276,129
4,PR,4879,349
5,SC,3531,190
6,BA,3275,19
7,DF,2072,30
8,ES,1964,23
9,GO,1950,40


Інтерпретація: Найбільше унікальних покупців представлено у штаті SP — 40296. У цьому ж штаті зареєстровано 1849 продавців. FULL OUTER JOIN дозволяє порівняти обидві групи без втрати штатів, які є лише в одній із таблиць.


In [25]:
# Query 4: Find pairs of orders belonging to the same customer
q4 = pd.read_sql(text("""
    SELECT
        c1.customer_unique_id,
        o1.order_id AS first_order_id,
        o1.order_purchase_timestamp AS first_order_date,
        o2.order_id AS second_order_id,
        o2.order_purchase_timestamp AS second_order_date
    FROM olist_customers_typed AS c1
    INNER JOIN olist_orders_typed AS o1
        ON o1.customer_id = c1.customer_id
    INNER JOIN olist_customers_typed AS c2
        ON c1.customer_unique_id = c2.customer_unique_id
       AND c1.customer_id <> c2.customer_id
    INNER JOIN olist_orders_typed AS o2
        ON o2.customer_id = c2.customer_id
       AND o1.order_id < o2.order_id
    ORDER BY
        c1.customer_unique_id,
        o1.order_purchase_timestamp
    LIMIT 10
"""), engine)

display(q4)

print(
    f"Інтерпретація: Запит показав {len(q4)} пар замовлень "
    "покупців, які робили покупки більше одного разу. "
    "SELF JOIN дозволив знайти різні customer_id, "
    "які належать одному customer_unique_id. "
    "Умова порівняння order_id допомагає уникнути "
    "повторення тих самих пар у зворотному порядку."
)

,customer_unique_id,first_order_id,first_order_date,second_order_id,second_order_date
0,00172711b30d52eea8b313a7f2cced02,bb874c45df1a3c97842d52f31efee99a,2018-07-28 00:23:49,c306eca42d32507b970739b5b6a5a33a,2018-08-13 09:14:07
1,004288347e5e88a27ded2bb23747066c,08204559bebd39e09ee52dcb56d8faa2,2018-01-14 07:36:54,a61d617fbe5bd006e40d3a0988fc844b,2017-07-27 14:13:03
2,004b45ec5c64187465168251cd1c9c2f,90ae229a4addcfead792e2564554f09c,2017-09-01 12:11:23,9392c5e72885ad5aba87e6223ca9838d,2018-05-26 19:42:48
3,0058f300f57d7b93c477a131a59b36c3,2cfc79d9582e9135c0a9b61fa60e6b21,2018-02-19 17:11:34,81a93b2fa39e104b865b2bc471c16008,2018-03-22 18:09:41
4,00a39521eb40f7012db50455bf083460,7d32c87acba91ed87ebd98310fe1c54d,2018-05-23 20:14:21,cea3e6c11eb60acb9d8d4d51694832f8,2018-06-03 10:12:57
5,00cc12a6d8b578b8ebd21ea4e2ae8b27,64307ceb91666760cf3ff463618302fd,2017-03-21 19:25:22,d61b915b69851aec8a8865f36cfd793e,2017-03-21 19:25:23
6,011575986092c30523ecb71ff10cb473,0f28d51fdd4828907bdc58b57f672e84,2018-02-17 15:54:49,ed0a6265b5b0a292cd537214f401f1fc,2018-04-18 21:58:08
7,011b4adcd54683b480c4d841250a987f,93b35affb86435b71ca01a6dd1eab2b1,2018-02-15 11:40:57,f05a68aaa0d8f89e758c7134d53fa22a,2017-08-22 12:51:29
8,012452d40dafae4df401bced74cdb490,ce2b4f2836d78829e4796213d536101e,2017-06-18 22:46:42,e49d0607832db7c3b1343764225d8df6,2018-05-14 12:12:45
9,012a218df8995d3ec3bb221828360c86,207258d556f9b9784b54de837ae49286,2018-06-18 13:08:38,e89e48d863049f7ca8a49758ae99f58a,2018-05-07 10:28:17


Інтерпретація: Запит показав 10 пар замовлень покупців, які робили покупки більше одного разу. SELF JOIN дозволив знайти різні customer_id, які належать одному customer_unique_id. Умова порівняння order_id допомагає уникнути повторення тих самих пар у зворотному порядку.


In [26]:
# Query 5: Find orders without reviews
q5 = pd.read_sql(text("""
    SELECT
        o.order_id,
        o.order_status,
        o.order_purchase_timestamp
    FROM olist_orders_typed AS o
    WHERE NOT EXISTS (
        SELECT 1
        FROM olist_order_reviews_typed AS r
        WHERE r.order_id = o.order_id
    )
    ORDER BY o.order_purchase_timestamp DESC NULLS LAST
    LIMIT 20
"""), engine)

display(q5)

,order_id,order_status,order_purchase_timestamp
0,afb61112fb99b07fe17e591c68c0c84c,canceled,2018-09-06 10:48:12
1,e91a3bc14b8d125d82f0334371c39926,delivered,2018-08-27 17:01:43
2,37e5725572d3cbd7aad103b730f9232a,delivered,2018-08-20 09:18:49
3,aa6e3cc7cf13ef1dc599d1243daee6f6,delivered,2018-08-18 23:54:05
4,4ac05edf248d6206469e17c32e52c9ac,delivered,2018-08-17 22:35:51
5,fea523ebb7e205d346f445544e0a18fd,delivered,2018-08-17 12:46:57
6,a46e7e8e02915adecb392ad065121a51,delivered,2018-08-17 10:59:18
7,bcbd3afc999b3fa9dbd61efb370239dd,delivered,2018-08-16 12:31:47
8,b94d017913e0b697201de9ec22c23f46,delivered,2018-08-16 10:11:23
9,ff5b7f440481674b38554434638beebf,delivered,2018-08-15 20:24:56


In [27]:
# Count all orders without reviews
q5_count = pd.read_sql(text("""
    SELECT
        COUNT(*) AS orders_without_reviews
    FROM olist_orders_typed AS o
    WHERE NOT EXISTS (
        SELECT 1
        FROM olist_order_reviews_typed AS r
        WHERE r.order_id = o.order_id
    )
"""), engine)

display(q5_count)

missing_reviews = q5_count.iloc[0]["orders_without_reviews"]

print(
    f"Інтерпретація: У датасеті знайдено "
    f"{missing_reviews} замовлень без відгуків. "
    "Для пошуку використано NOT EXISTS, який перевіряє "
    "відсутність відповідного запису в таблиці reviews. "
    "Такі замовлення можуть бути важливими для аналізу "
    "якості обслуговування, але відсутність відгуку "
    "не обов'язково означає незадоволеність покупця."
)

,orders_without_reviews
0,768


Інтерпретація: У датасеті знайдено 768 замовлень без відгуків. Для пошуку використано NOT EXISTS, який перевіряє відсутність відповідного запису в таблиці reviews. Такі замовлення можуть бути важливими для аналізу якості обслуговування, але відсутність відгуку не обов'язково означає незадоволеність покупця.


### Висновок

У цьому завданні було виконано п'ять SQL-запитів із різними типами `JOIN`.

За допомогою `INNER JOIN` було проаналізовано кількість замовлень у різних штатах Бразилії. `LEFT JOIN` використано для аналізу продажів товарів, а `FULL OUTER JOIN` - для порівняння кількості покупців і продавців за штатами.

Через `SELF JOIN` було знайдено покупців, які мають більше одного замовлення. Також за допомогою anti-join було перевірено, які замовлення не мають відгуків.

Під час роботи важливо було враховувати зв'язки між таблицями. Наприклад, одне замовлення може містити кілька товарів або відгуків, тому неправильний `JOIN` може призвести до дублювання результатів.

Для уникнення таких помилок у деяких запитах дані спочатку групувалися, а вже після цього з'єднувалися між собою.

## Task 5. Aggregations and Business Metrics

In [28]:
# Query 1: Calculate seller ratings without multiplying reviews
q1 = pd.read_sql(text("""
    WITH seller_orders AS (
        SELECT DISTINCT
            seller_id,
            order_id
        FROM olist_order_items_typed
    ),
    order_reviews AS (
        SELECT
            order_id,
            AVG(review_score::NUMERIC) AS order_score
        FROM olist_order_reviews_typed
        GROUP BY order_id
    )
    SELECT
        so.seller_id,
        COUNT(*) AS total_orders,
        COUNT(r.order_score) AS reviewed_orders,
        ROUND(AVG(r.order_score), 2) AS seller_score
    FROM seller_orders AS so
    LEFT JOIN order_reviews AS r
        ON so.order_id = r.order_id
    GROUP BY so.seller_id
    HAVING COUNT(r.order_score) >= 10
    ORDER BY seller_score DESC, reviewed_orders DESC, so.seller_id
    LIMIT 10
"""), engine)

display(q1)

,seller_id,total_orders,reviewed_orders,seller_score
0,48efc9d94a9834137efd9ea76b065a38,33,33,5.00
1,2addf05f476d0637864454e93ba673d5,12,12,5.00
2,41c2bad7229b0c25e6becf179ebf63ff,20,20,4.95
3,b1fecf4da1fa2689bccffa0121953643,19,19,4.95
4,2d518637f53161b973e01f56ea4bb88e,13,13,4.92
5,9d681c7e12db302cb261e721040dde65,13,13,4.92
6,334cab711dee080b079fa5779b584783,11,11,4.91
7,788e857f317e53de488d393e65a80f45,11,11,4.91
8,d4910f1cdcfdabd48d6b316e395d4a23,11,11,4.91
9,2f4b0d3b9634b647d4edee577d7ebb7e,10,10,4.90


In [29]:
# Interpret seller rating results
if not q1.empty:
    top = q1.iloc[0]

    print(
        f"Інтерпретація: Найвищу середню оцінку серед вибраних "
        f"продавців має {top['seller_id']} — {top['seller_score']}. "
        f"Для нього враховано {top['reviewed_orders']} оцінених замовлень. "
        "У розрахунку кожне замовлення враховується один раз, "
        "тому кількість товарів не впливає на вагу оцінки."
    )
else:
    print("Продавців із 10 оціненими замовленнями не знайдено.")

Інтерпретація: Найвищу середню оцінку серед вибраних продавців має 48efc9d94a9834137efd9ea76b065a38 — 5.0. Для нього враховано 33 оцінених замовлень. У розрахунку кожне замовлення враховується один раз, тому кількість товарів не впливає на вагу оцінки.


In [30]:
# Create a table for materialized seller metrics
seller_score_ddl = """
DROP TABLE IF EXISTS seller_score CASCADE;

CREATE TABLE seller_score (
    seller_id TEXT PRIMARY KEY
        REFERENCES olist_sellers_typed(seller_id),
    total_orders INTEGER NOT NULL
        CHECK (total_orders >= 0),
    reviewed_orders INTEGER NOT NULL
        CHECK (reviewed_orders >= 0),
    avg_review_score NUMERIC(4, 2)
        CHECK (
            avg_review_score IS NULL
            OR avg_review_score BETWEEN 1 AND 5
        ),
    updated_at TIMESTAMPTZ NOT NULL DEFAULT NOW(),

    CHECK (reviewed_orders <= total_orders)
);
"""

with engine.begin() as conn:
    conn.execute(text(seller_score_ddl))

print("seller_score table created successfully.")

seller_score table created successfully.


In [31]:
# Populate seller_score using order-level reviews
seller_score_insert = """
WITH seller_orders AS (
    SELECT DISTINCT
        seller_id,
        order_id
    FROM olist_order_items_typed
),
order_reviews AS (
    SELECT
        order_id,
        AVG(review_score::NUMERIC) AS order_score
    FROM olist_order_reviews_typed
    GROUP BY order_id
),
seller_stats AS (
    SELECT
        so.seller_id,
        COUNT(*)::INTEGER AS total_orders,
        COUNT(r.order_score)::INTEGER AS reviewed_orders,
        ROUND(AVG(r.order_score), 2) AS avg_review_score
    FROM seller_orders AS so
    LEFT JOIN order_reviews AS r
        ON so.order_id = r.order_id
    GROUP BY so.seller_id
)
INSERT INTO seller_score (
    seller_id,
    total_orders,
    reviewed_orders,
    avg_review_score
)
SELECT
    seller_id,
    total_orders,
    reviewed_orders,
    avg_review_score
FROM seller_stats

ON CONFLICT (seller_id)
DO UPDATE SET
    total_orders = EXCLUDED.total_orders,
    reviewed_orders = EXCLUDED.reviewed_orders,
    avg_review_score = EXCLUDED.avg_review_score,
    updated_at = NOW();
"""

with engine.begin() as conn:
    conn.execute(text(seller_score_insert))

print("Seller scores calculated successfully.")

Seller scores calculated successfully.


In [32]:
# Verify seller score records
df = pd.read_sql(text("""
    SELECT
        COUNT(*) AS scored_sellers,
        COUNT(avg_review_score) AS sellers_with_reviews,
        ROUND(AVG(avg_review_score), 2) AS mean_seller_score
    FROM seller_score
"""), engine)

display(df)

,scored_sellers,sellers_with_reviews,mean_seller_score
0,3095,3090,4.0


In [33]:
# Query 2: Analyze sales by product category
q2 = pd.read_sql(text("""
    SELECT
        COALESCE(p.product_category_name, 'Unknown') AS category,
        COUNT(*) AS items_sold,
        ROUND(SUM(oi.price), 2) AS total_sales,
        ROUND(AVG(oi.price), 2) AS avg_price
    FROM olist_order_items_typed AS oi
    INNER JOIN olist_products_typed AS p
        ON oi.product_id = p.product_id
    GROUP BY COALESCE(p.product_category_name, 'Unknown')
    HAVING COUNT(*) >= 100
    ORDER BY total_sales DESC, category
    LIMIT 10
"""), engine)

display(q2)

top = q2.iloc[0]

print(
    f"Інтерпретація: Найбільшу суму продажів серед вибраних "
    f"категорій має {top['category']} — {top['total_sales']}. "
    f"У цій категорії продано {top['items_sold']} позицій, "
    f"а середня ціна становить {top['avg_price']}. "
    "Для аналізу використано тільки категорії "
    "зі щонайменше 100 проданими позиціями."
)

,category,items_sold,total_sales,avg_price
0,beleza_saude,9670,1258681.34,130.16
1,relogios_presentes,5991,1205005.68,201.14
2,cama_mesa_banho,11115,1036988.68,93.30
3,esporte_lazer,8641,988048.97,114.34
4,informatica_acessorios,7827,911954.32,116.51
5,moveis_decoracao,8334,729762.49,87.56
6,cool_stuff,3796,635290.85,167.36
7,utilidades_domesticas,6964,632248.66,90.79
8,automotivo,4235,592720.11,139.96
9,ferramentas_jardim,4347,485256.46,111.63


Інтерпретація: Найбільшу суму продажів серед вибраних категорій має beleza_saude — 1258681.34. У цій категорії продано 9670 позицій, а середня ціна становить 130.16. Для аналізу використано тільки категорії зі щонайменше 100 проданими позиціями.


In [34]:
# Query 3: Count sellers and combine city names by state
q3 = pd.read_sql(text("""
    SELECT
        seller_state,
        COUNT(*) AS seller_count,
        COUNT(DISTINCT seller_city) AS city_count,
        LEFT(
            STRING_AGG(
                DISTINCT seller_city,
                ', ' ORDER BY seller_city
            ),
            150
        ) AS cities_sample
    FROM olist_sellers_typed
    GROUP BY seller_state
    HAVING COUNT(*) >= 20
    ORDER BY seller_count DESC, seller_state
    LIMIT 10
"""), engine)

display(q3)

top = q3.iloc[0]

print(
    f"Інтерпретація: Найбільше продавців серед вибраних "
    f"штатів знаходиться у {top['seller_state']} — "
    f"{top['seller_count']}. "
    f"Вони представлені у {top['city_count']} різних містах. "
    "STRING_AGG дозволяє отримати список міст в одному полі. "
    "Для зручності відображено лише початок цього списку."
)

,seller_state,seller_count,city_count,cities_sample
0,SP,1849,261,"aguas claras df, alambari, alvares machado, am..."
1,PR,349,67,"almirante tamandare, ampere, andira-pr, apucar..."
2,MG,244,82,"alfenas, andradas, araguari, araxa, arinos, ba..."
3,SC,190,65,"araquari, ararangua, balenario camboriu, balne..."
4,RJ,171,38,"04482255, angra dos reis, angra dos reis rj, a..."
5,RS,129,51,"alvorada, arvorezinha, bage, bento goncalves, ..."
6,GO,40,12,"abadia de goias, anapolis, aparecida de goiani..."
7,DF,30,3,"brasilia, brasilia df, gama"
8,ES,23,11,"afonso claudio, cachoeiro de itapemirim, caria..."


Інтерпретація: Найбільше продавців серед вибраних штатів знаходиться у SP — 1849. Вони представлені у 261 різних містах. STRING_AGG дозволяє отримати список міст в одному полі. Для зручності відображено лише початок цього списку.


In [35]:
# Validate seller score aggregation
df = pd.read_sql(text("""
    SELECT
        COUNT(*) AS total_sellers,
        COUNT(DISTINCT seller_id) AS unique_sellers,
        COUNT(*) FILTER (
            WHERE avg_review_score IS NOT NULL
              AND avg_review_score NOT BETWEEN 1 AND 5
        ) AS invalid_scores,
        COUNT(*) FILTER (
            WHERE reviewed_orders > total_orders
        ) AS invalid_order_counts
    FROM seller_score
"""), engine)

display(df)

assert df.iloc[0]["total_sellers"] == df.iloc[0]["unique_sellers"]
assert df.iloc[0]["invalid_scores"] == 0
assert df.iloc[0]["invalid_order_counts"] == 0

print("Seller score validation passed.")

,total_sellers,unique_sellers,invalid_scores,invalid_order_counts
0,3095,3095,0,0


Seller score validation passed.


### Висновок

У цьому завданні було використано `GROUP BY`, `HAVING` та агрегатні функції для розрахунку бізнес-метрик.

Було розраховано середні оцінки продавців, суму продажів за категоріями товарів та кількість продавців у різних штатах.

Для розрахунку `seller_score` спочатку було згруповано дані за продавцем і замовленням. Це дозволило уникнути повторного врахування відгуків, якщо замовлення містить кілька товарів. Якщо замовлення мало кілька відгуків, спочатку розраховувалася їхня середня оцінка.

У запитах також використано `HAVING` для відбору груп за кількістю записів, а `STRING_AGG` - для об'єднання назв міст.

Отримані метрики можна використовувати для аналізу продажів, роботи продавців та подальшої підготовки даних для ML.

## Task 6. Set Operations, Window Functions and Reflection

In [36]:
# Compare customer groups using INTERSECT and EXCEPT
q_set = pd.read_sql(text("""
    WITH beauty_customers AS (
        SELECT DISTINCT c.customer_unique_id
        FROM olist_customers_typed AS c
        JOIN olist_orders_typed AS o
            ON c.customer_id = o.customer_id
        JOIN olist_order_items_typed AS oi
            ON o.order_id = oi.order_id
        JOIN olist_products_typed AS p
            ON oi.product_id = p.product_id
        WHERE p.product_category_name = 'beleza_saude'
    ),
    tech_customers AS (
        SELECT DISTINCT c.customer_unique_id
        FROM olist_customers_typed AS c
        JOIN olist_orders_typed AS o
            ON c.customer_id = o.customer_id
        JOIN olist_order_items_typed AS oi
            ON o.order_id = oi.order_id
        JOIN olist_products_typed AS p
            ON oi.product_id = p.product_id
        WHERE p.product_category_name = 'informatica_acessorios'
    )
    SELECT
        'Both categories' AS customer_group,
        COUNT(*) AS customer_count
    FROM (
        SELECT customer_unique_id FROM beauty_customers
        INTERSECT
        SELECT customer_unique_id FROM tech_customers
    ) AS both_categories

    UNION ALL

    SELECT
        'Only beauty' AS customer_group,
        COUNT(*) AS customer_count
    FROM (
        SELECT customer_unique_id FROM beauty_customers
        EXCEPT
        SELECT customer_unique_id FROM tech_customers
    ) AS only_beauty
"""), engine)

display(q_set)

,customer_group,customer_count
0,Both categories,15
1,Only beauty,8663


In [37]:
# Interpret the set operation results
both = int(q_set.loc[
    q_set["customer_group"] == "Both categories",
    "customer_count"
].iloc[0])

only_beauty = int(q_set.loc[
    q_set["customer_group"] == "Only beauty",
    "customer_count"
].iloc[0])

print(
    f"Інтерпретація: Товари з обох категорій купували "
    f"{both} покупців. "
    f"Ще {only_beauty} покупців замовляли товари з категорії "
    "краси та здоров'я, але не купували комп'ютерні аксесуари. "
    "Таким чином, set operations дозволяють порівнювати "
    "різні групи покупців без дублювання записів."
)

Інтерпретація: Товари з обох категорій купували 15 покупців. Ще 8663 покупців замовляли товари з категорії краси та здоров'я, але не купували комп'ютерні аксесуари. Таким чином, set operations дозволяють порівнювати різні групи покупців без дублювання записів.


In [38]:
# Rank sellers within each state using a window function
q_window = pd.read_sql(text("""
    WITH ranked_sellers AS (
        SELECT
            s.seller_id,
            s.seller_state,
            sc.reviewed_orders,
            sc.avg_review_score,

            DENSE_RANK() OVER (
                PARTITION BY s.seller_state
                ORDER BY sc.avg_review_score DESC
            ) AS state_rank

        FROM olist_sellers_typed AS s
        INNER JOIN seller_score AS sc
            ON s.seller_id = sc.seller_id

        WHERE sc.reviewed_orders >= 10
          AND sc.avg_review_score IS NOT NULL
    )

    SELECT
        seller_id,
        seller_state,
        reviewed_orders,
        avg_review_score,
        state_rank
    FROM ranked_sellers
    WHERE state_rank <= 3
    ORDER BY
        seller_state,
        state_rank,
        reviewed_orders DESC,
        seller_id
    LIMIT 30
"""), engine)

display(q_window)

,seller_id,seller_state,reviewed_orders,avg_review_score,state_rank
0,a3dd39f583bc80bd8c5901c95878921e,BA,34,4.59,1
1,d03698c2efd04a549382afa6623e27fb,BA,18,4.50,2
2,c72de06d72748d1a0dfb2125be43ba63,BA,21,4.48,3
3,bbf9ad41dca6603e614efcdad7aab8c4,CE,14,4.50,1
4,8d79c8a04e42d722a75097ce5cbcf2ef,CE,27,4.44,2
5,dbdd0ec73a4817971d962698f2fea022,CE,16,4.38,3
6,2addf05f476d0637864454e93ba673d5,DF,12,5.00,1
7,77e4abfd739aed9c46480f90f81c0eb4,DF,18,4.78,2
8,89a51f50b8095ea78d5768f34c13a76f,DF,10,4.70,3
9,41da412d33e8da4f22baf55cb1bde82c,ES,15,4.67,1


In [39]:
# Interpret the seller ranking results
if not q_window.empty:
    best = q_window.iloc[0]

    print(
        f"Інтерпретація: У штаті {best['seller_state']} "
        f"продавець {best['seller_id']} має середню оцінку "
        f"{best['avg_review_score']} та рейтинг "
        f"{best['state_rank']} у своєму штаті. "
        "DENSE_RANK дозволяє порівнювати продавців окремо "
        "в кожному штаті. Якщо кілька продавців мають "
        "однакову оцінку, вони отримують однаковий рейтинг."
    )
else:
    print("Продавців, які відповідають умовам, не знайдено.")

Інтерпретація: У штаті BA продавець a3dd39f583bc80bd8c5901c95878921e має середню оцінку 4.59 та рейтинг 1 у своєму штаті. DENSE_RANK дозволяє порівнювати продавців окремо в кожному штаті. Якщо кілька продавців мають однакову оцінку, вони отримують однаковий рейтинг.


In [40]:
# Verify set operations and seller rankings
assert len(q_set) == 2
assert (q_set["customer_count"] >= 0).all()

assert not q_window.empty
assert q_window["state_rank"].between(1, 3).all()
assert (q_window["reviewed_orders"] >= 10).all()
assert q_window["avg_review_score"].between(1, 5).all()

print("Set operations and window function checks passed.")

Set operations and window function checks passed.


## 7. Reflection

У цьому домашньому завданні було розглянуто роботу з кількома пов'язаними таблицями на прикладі датасету Olist.

**1. Робота з JOIN**

Найважливішим було правильно з'єднувати таблиці, оскільки вони містять дані різного рівня деталізації. Наприклад, одне замовлення може мати декілька товарів і відгуків. Якщо просто з'єднати всі таблиці, можна отримати дублікати та неправильні результати обчислень. Тому для розрахунку оцінок продавців спочатку було згруповано дані за продавцем і замовленням.

**2. DML та DDL**

Було створено таблиці з первинними й зовнішніми ключами та обмеженнями для перевірки даних. Також було використано `INSERT`, `UPDATE` і `DELETE`. Особливо корисною виявилася команда `ON CONFLICT`, оскільки вона дозволяє повторно запускати завантаження без створення дублікатів.

**3. Window Functions**

За допомогою `DENSE_RANK()` було побудовано рейтинг продавців за середніми оцінками. Це зручно, оскільки можна порівнювати продавців окремо в кожному штаті, не об'єднуючи їх в один рядок.

**4. Якість даних**

Важливо враховувати пропущені значення, повторні записи та правильні зв'язки між таблицями. Наприклад, для відгуків було використано окремий технічний ключ, оскільки `review_id` не обов'язково унікальний.

**5. Сегментація покупців**

Для сегментації було використано `customer_unique_id`, а не `customer_id`. Це дозволило врахувати всі покупки однієї людини разом і правильно розрахувати кількість замовлень та загальні витрати.

Загалом це завдання допомогло краще зрозуміти, як структура бази даних впливає на правильність аналітичних результатів.